In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os 
from matplotlib.ticker import AutoMinorLocator
import thermoift.PLOT_SETTINGS as ps

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

try:
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion
except ImportError as exc:
    raise ImportError(
        "tabpfn is not installed. Run `%pip install tabpfn` in a notebook cell, then rerun."
    ) from exc

In [2]:
csv_path = "interfacial_results_dataset_A1.csv"
combined_df = pd.read_csv(csv_path)

print(f"Loaded {csv_path} with shape: {combined_df.shape}")
combined_df.head()

Loaded interfacial_results_dataset_A1.csv with shape: (19171, 74)


,temperature,pressure,z_carbon dioxide,z_hydrogen,z_nitrogen,z_argon,z_methane,z_oxygen,z_water,z_carbon monoxide,...,E_sulfur dioxide,E_hydrogen sulfide,E_propane,E_ethane,gamma0_CO2,rhoL0_CO2,rhoV0_CO2,Tc_CO2,Psat_CO2,source_id
0,200.0,2.523496,0.95,0.005733,0.0,0.0,0.0,0.016922,0.0,0.027345,...,0.0,0.0,0.0,0.0,19.912634,1222.376153,6.589392,309.147376,2.392259,0
1,200.0,6.239821,0.95,0.005733,0.0,0.0,0.0,0.016922,0.0,0.027345,...,0.0,0.0,0.0,0.0,19.912634,1222.376153,6.589392,309.147376,2.392259,0
2,200.0,9.956146,0.95,0.005733,0.0,0.0,0.0,0.016922,0.0,0.027345,...,0.0,0.0,0.0,0.0,19.912634,1222.376153,6.589392,309.147376,2.392259,0
3,200.0,13.672471,0.95,0.005733,0.0,0.0,0.0,0.016922,0.0,0.027345,...,0.0,0.0,0.0,0.0,19.912634,1222.376153,6.589392,309.147376,2.392259,0
4,200.0,17.388795,0.95,0.005733,0.0,0.0,0.0,0.016922,0.0,0.027345,...,0.0,0.0,0.0,0.0,19.912634,1222.376153,6.589392,309.147376,2.392259,0


In [3]:
target = "gamma"

COMPONENTS = [
    "carbon dioxide",
    "hydrogen",
    "argon",
    "nitrogen",
    "methane",
    "oxygen",
    "carbon monoxide",
    "hydrogen sulfide",
]

z_columns = [f"z_{comp}" for comp in COMPONENTS]
feature_cols = [
    col for col in combined_df.columns
    if col in ["temperature", "pressure"] + z_columns
]

required_cols = feature_cols + [target]
df_model = combined_df[required_cols].dropna().copy()

X = df_model[feature_cols]
y = df_model[target]

print("Selected features:", feature_cols)
print(f"Rows after dropna on features + target: {len(df_model)}")

Selected features: ['temperature', 'pressure', 'z_carbon dioxide', 'z_hydrogen', 'z_nitrogen', 'z_argon', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']
Rows after dropna on features + target: 19171


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=85
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows:  {X_test.shape[0]}")

Training rows: 15336
Testing rows:  3835


In [5]:
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

tabpfn_model = TabPFNRegressor(
    random_state=85,
    ignore_pretraining_limits=True,
    fit_mode="fit_preprocessors",
)
tabpfn_model.fit(X_train, y_train)

y_pred_train = tabpfn_model.predict(X_train)
y_pred_test = tabpfn_model.predict(X_test)

print("TabPFN model training completed.")

TabPFN model training completed.


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

train_metrics = regression_metrics(y_train, y_pred_train)
test_metrics = regression_metrics(y_test, y_pred_test)

metrics_df = pd.DataFrame([train_metrics, test_metrics], index=["Train", "Test"])
print("=" * 60)
print("TabPFN Model Performance for gamma")
print("=" * 60)
print(f"\nTraining Set:")
print(f"  R²:   {train_metrics['R2']:.6f}")
print(f"  RMSE: {train_metrics['RMSE']:.6f} mN/m")
print(f"  MAE:  {train_metrics['MAE']:.6f} mN/m")
print(f"\nTest Set:")
print(f"  R²:   {test_metrics['R2']:.6f}")
print(f"  RMSE: {test_metrics['RMSE']:.6f} mN/m")
print(f"  MAE:  {test_metrics['MAE']:.6f} mN/m")
metrics_df

In [ ]:
# Parity plot
fig, ax = ps.plot_init()

lims = [
    min(y_test.min(), np.min(y_pred_test)),
    max(y_test.max(), np.max(y_pred_test))
]
lims = [lims[0] - 0.5, lims[1] + 0.5]

ax.scatter(
    y_test, y_pred_test,
    alpha=0.8, s=20,
    facecolors="lightgreen",
    edgecolors="darkgreen",
    linewidths=0.6,
    label="TabPFN"
)

ax.plot(lims, lims, "k--", linewidth=1.4, label="Perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel(r"Actual $\gamma$ / [mN m$^{-1}$]", fontsize=ps.label_fontsize * 0.9)
ax.set_ylabel(r"Predicted $\gamma$ / [mN m$^{-1}$]", fontsize=ps.label_fontsize * 0.9)
ax.set_title(r"TabPFN Parity Plot (Test Set)", fontsize=ps.title_fontsize * 0.75, fontweight="bold")

ps.apply_axis_style(ax)
# dummy_r2 = Line2D([], [], linestyle="none", label=rf"$R^2$ = {test_metrics['R2']:.4f}")

handles, labels = ax.get_legend_handles_labels()
# handles.append(dummy_r2)

ax.legend(
    handles=handles,
    fontsize=ps.label_fontsize * 0.65,
    loc="upper left",
    edgecolor="black",
    framealpha=1.0
)

ax.minorticks_on()
ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))
ax.tick_params(axis="both", which="minor", length=3)

plt.tight_layout()
ps.save_plot(fig, "tabpfn_parity_plot_Gamma")
plt.show()

In [ ]:
# Residual distribution
residuals = y_test.values - y_pred_test

fig, ax = ps.plot_init()

ax.hist(
    residuals,
    bins=40,
    alpha=0.75,
    color="lightgreen",
    edgecolor="darkgreen",
    linewidth=0.8,
    label="Residuals"
)

ax.axvline(0, color="black", linewidth=1.3, linestyle="--", label="Zero residual")

mean_resid = residuals.mean()
std_resid = residuals.std()

ax.set_xlabel(r"Residual / [mN m$^{-1}$]", fontsize=ps.label_fontsize * 0.9)
ax.set_ylabel(r"Count", fontsize=ps.label_fontsize * 0.9)
ax.set_title(r"Residual Distribution", fontsize=ps.title_fontsize * 0.75, fontweight="bold")

# dummy_mean = Line2D([], [], linestyle="none", label=rf"Mean = {mean_resid:.4f} mN/m")
# dummy_std  = Line2D([], [], linestyle="none", label=rf"Std Dev = {std_resid:.4f} mN/m")
# dummy_rmse = Line2D([], [], linestyle="none", label=rf"RMSE = {test_metrics['RMSE']:.4f} mN/m")

handles, labels = ax.get_legend_handles_labels()
# handles.extend([dummy_mean, dummy_std, dummy_rmse])

ps.apply_axis_style(ax)

ax.legend(
    handles=handles,
    fontsize=ps.label_fontsize * 0.5,
    loc="upper left",
    edgecolor="black",
    facecolor="white",
    framealpha=1.0
)

ax.minorticks_on()
ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))
ax.tick_params(axis="both", which="minor", length=3)

plt.tight_layout()
ps.save_plot(fig, "tabpfn_Gamma_residual_distribution")
plt.show()

In [ ]:
# Residuals vs Predicted
fig, ax = ps.plot_init()

ax.scatter(y_pred_test, residuals, alpha=0.8, s=20, facecolors="lightgreen", edgecolors="darkgreen", linewidths=0.6, label="Residuals")

ax.axhline(0, color="black", linewidth=1.3, linestyle="--", label="Zero residual")
ax.axhline(test_metrics['RMSE'], color="blue", linewidth=1.1, linestyle=":", alpha=0.85, label=rf"$\pm$RMSE = {test_metrics['RMSE']:.4f} mN/m")
ax.axhline(-test_metrics['RMSE'], color="blue", linewidth=1.1, linestyle=":", alpha=0.85)

ax.set_xlabel(r"Predicted $\gamma$ / [mN m$^{-1}$]", fontsize=ps.label_fontsize * 0.9)
ax.set_ylabel(r"Residual / [mN m$^{-1}$]", fontsize=ps.label_fontsize * 0.9)

ps.apply_axis_style(ax)
ps.style_legend(ax, fontsize=ps.label_fontsize * 0.5, loc="upper left")

ax.minorticks_on()
ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))
ax.tick_params(axis="both", which="minor", length=3)

plt.tight_layout()
ps.save_plot(fig, "tabpfn_Gamma_residual_vs_predicted")
plt.show()

In [8]:
pred_df = X_test.copy()
pred_df["gamma_actual"] = y_test.values
pred_df["gamma_pred_tabpfn"] = y_pred_test

pred_df.to_csv("tabpfn_test_predictions.csv", index=False)
print("Saved predictions to tabpfn_test_predictions.csv")
pred_df.head()

Saved predictions to tabpfn_test_predictions.csv


,temperature,pressure,z_carbon dioxide,z_hydrogen,z_nitrogen,z_argon,z_methane,z_oxygen,z_carbon monoxide,z_hydrogen sulfide,gamma_actual,gamma_pred_tabpfn
4330,275.0,37.256439,0.990000,0.000000,0.000000,0.000000,0.010000,0.000000,0.000000,0.0,4.105066,4.103269
13242,245.0,15.527731,0.980000,0.014697,0.000000,0.000000,0.002563,0.000333,0.002407,0.0,9.859954,9.870987
8506,250.0,19.386534,0.990000,0.000000,0.000000,0.003023,0.003007,0.003970,0.000000,0.0,8.646487,8.647531
12605,210.0,8.725639,0.949999,0.000410,0.003068,0.000000,0.024408,0.022115,0.000000,0.0,15.886195,15.884888
13969,220.0,7.891495,0.960000,0.000000,0.000000,0.030000,0.000000,0.000000,0.010000,0.0,14.906069,14.899911


In [ ]:
# 5-fold Cross-Validation
print("Running 5-fold cross-validation (this may take a while)...")
cv_scores = cross_val_score(tabpfn_model, X, y, cv=5, scoring='r2')

print("\n" + "=" * 60)
print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 60)
print(f"Cross-Validation R² Scores: {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

In [ ]:
print("=" * 70)
print("SUMMARY: TabPFN Model for gamma Prediction")
print("=" * 70)
print(f"\nModel: TabPFNRegressor")
print(f"Features used: {feature_cols}")
print(f"\nPerformance Metrics (Training Set):")
print(f"  - R²:   {train_metrics['R2']:.6f}")
print(f"  - RMSE: {train_metrics['RMSE']:.6f} mN/m")
print(f"  - MAE:  {train_metrics['MAE']:.6f} mN/m")
print(f"\nPerformance Metrics (Test Set):")
print(f"  - R²:   {test_metrics['R2']:.6f}")
print(f"  - RMSE: {test_metrics['RMSE']:.6f} mN/m")
print(f"  - MAE:  {test_metrics['MAE']:.6f} mN/m")
print(f"\nCross-Validation:")
print(f"  - Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")